In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import scoring
import helpers

In [3]:
resumes = pd.read_parquet('processed/resume_embeddings.parquet')

In [4]:
jobs = pd.read_parquet('processed/dice_job_descriptions_embeddings.parquet')

In [5]:
resumes

,ID,Resume_str,Resume_html,Category,extracted_skills,length,matched_skills,matched_skills_ordered,skill_embeddings,skill_embeddings_ordered,avg_skill_embedding
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR,"[Accounting, Ads, Advertising, Analytical skil...",5442,"[accounting, create advertisements, outdoor ad...","[accounting, company policies, conduct public ...","[-0.051119804, 0.051380936, -0.1956489, -0.000...","[-0.028823888, 0.056566164, -0.19899392, -0.00...","[0.0002473207458333337, 0.03493942622916666, -..."
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR,"[Adobe Photoshop, ADP, Asset Management, brand...",5572,"[Adobe Photoshop, asset management, maintain c...","[Adobe Illustrator, Adobe Photoshop, asset man...","[-0.013715399, 0.01982399, -0.15967667, 0.0198...","[-0.02063493, 0.030683268, -0.16921557, 0.0126...","[0.002284706435294118, 0.01609132022352941, -0..."
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR,"[Recruiting, FMLA/EEO/FLSA, HRIS Development, ...",7720,"[recruit members, foreign affairs policy devel...","[Microsoft Access, desktop publishing, foreign...","[-0.037161566, 0.0036097213, -0.19848743, -0.0...","[0.0076125017, 0.01415186, -0.17561796, -0.009...","[-0.0002954738888888898, 0.02081807622222222, ..."
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR,"[Type 50 wpm and 10-Key by touch, Microsoft pr...",2855,"[personnel management, customer service, resol...","[customer service, personnel management, resol...","[-0.004071229, -0.0062129777, -0.16848274, -0....","[-0.0024955925, -0.005882781, -0.17091864, -0....","[0.007837255066666667, 0.0018740463333333332, ..."
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR,"[ADA, ADP, art, agency, benefits, Benefits Adm...",9172,"[create artwork, coaching techniques, coaching...","[Microsoft Access, Microsoft Visio, coaching t...","[0.008319367, 0.02057786, -0.17201392, 0.01393...","[4.8317004e-05, 0.005026085, -0.17149556, 0.01...","[0.00895575635652174, 0.013967274808695652, -0..."
...,...,...,...,...,...,...,...,...,...,...,...
2479,99416532,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION,"[Secret Clearance, Stock Control, Management, ...",5533,"[follow stock control instructions, operations...","[communication, follow stock control instructi...","[0.0049826736, 0.0045687603, -0.19545835, -0.0...","[0.0075103026, -0.0060989116, -0.19636096, -0....","[0.007711735449999999, 0.005379060733333334, -..."
2480,24589765,"GOVERNMENT RELATIONS, COMMUNICATIONS ...","<div class=""fontsize fontface vmargins hmargin...",AVIATION,"[arbitration, agency, budgets, Budget, continu...",7108,"[examine budgets, update budget, continuous im...","[Microsoft Visio, computer graphics, continuou...","[0.04153785, 0.038161203, -0.16648234, -0.0115...","[0.048209656, 0.025236994, -0.15883973, -0.005...","[0.013527077672727272, 0.028311003981818184, -..."
2481,31605080,GEEK SQUAD AGENT Professional...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION,"[Active Directory, Hardware, Customer Service,...",2020,"[hardware materials, customer service, manage ...","[assemble windows, customer service, hardware ...","[0.010476048, 0.028934365, -0.16972545, 0.0036...","[0.010129859, 0.025989288, -0.16765723, 0.0025...","[0.025738319725, 0.032416258287500006, -0.1611..."
2482,21190805,PROGRAM DIRECTOR / OFFICE MANAGER ...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION,"[Adobe, CPR, Customer Service, Customer Care, ...",5074,"[customer service, maintain customer service, ...","[Microsoft Access, chair a meeting, customer s...","[-0.013092378, 0.055586472, -0.17801957, -0.01...","[-0.006703326, 0.049669117, -0.16404499, -0.01...","[-0.003938775142857142,

In [6]:
resumes = resumes[~resumes['avg_skill_embedding'].isna()]

In [7]:
resumes = resumes.reset_index(drop=True)

In [8]:
jobs

,advertiserurl,company,employmenttype_jobstatus,jobdescription,jobid,joblocation_address,jobtitle,postdate,shift,site_name,skills,uniq_id,extracted_skills,matched_skills,matched_skills_ordered,skill_embeddings,skill_embeddings_ordered,avg_skill_embedding
0,https://www.dice.com/jobs/detail/AUTOMATION-TE...,"Digital Intelligence Systems, LLC","C2H Corp-To-Corp, C2H Independent, C2H W2, 3 M...",Looking for Selenium engineers...must have sol...,Dice Id : 10110693,"Atlanta, GA",AUTOMATION TEST ENGINEER,1 hour ago,Telecommuting not available|Travel not required,None,see below,418ff92580b270ef4e7c14f0ddfc36b4,"[Selenium, Java, Data Structures, Object Orien...","[Java (computer programming), information stru...","[Groovy, Java (computer programming), Oracle R...","[0.00816865, 0.04228201, -0.12763296, -0.06408...","[0.0012203407, 0.048476767, -0.1365127, -0.069...","[0.012229671666666667, 0.02882406688888889, -0..."
1,https://www.dice.com/jobs/detail/Information-S...,University of Chicago/IT Services,Full Time,The University of Chicago has a rapidly growin...,Dice Id : 10114469,"Chicago, IL",Information Security Engineer,1 week ago,Telecommuting not available|Travel not required,None,"linux/unix, network monitoring, incident respo...",8aec88cba08d53da65ab99cf20f6f9d9,"[Incident Response, Information Security Asses...","[investigate security issues, documentation ty...","[documentation types, investigate security iss...","[0.018865215, 0.036568727, -0.16978557, 0.0041...","[0.018206175, 0.013558769, -0.1604094, 0.00365...","[0.02598377266666667, 0.018216321833333337, -0..."
2,https://www.dice.com/jobs/detail/Business-Solu...,"Galaxy Systems, Inc.",Full Time,"GalaxE.SolutionsEvery day, our solutions affec...",Dice Id : CXGALXYS,"Schaumburg, IL",Business Solutions Architect,2 weeks ago,Telecommuting not available|Travel not required,None,"enterprise solutions architecture, business in...",46baa1f69ac07779274bcd90b85d9a72,"[Business Intelligence, Data Analysis, Data Wa...","[business intelligence, perform data analysis,...","[apply change management, business intelligenc...","[0.011157077, -0.003909247, -0.17394324, 0.010...","[0.011065557, -0.012342164, -0.18372022, 0.000...","[0.007513185861666667, -0.007068083333333332, ..."
3,https://www.dice.com/jobs/detail/Java-Develope...,TransTech LLC,Full Time,Java DeveloperFull-time/direct-hireBolingbrook...,Dice Id : 10113627,"Bolingbrook, IL","Java Developer (mid level)- FT- GREAT culture,...",2 weeks ago,Telecommuting not available|Travel not required,None,please see job description,3941b2f206ae0f900c4fba4ac0b18719,"[Java, JDBC, Multithreading, Linux/AIX/Unix, S...","[Java (computer programming), SQL, information...","[Java (computer programming), SQL, communicati...","[0.000571077, 0.026021862, -0.13816436, -0.033...","[0.00030429946, 0.02412445, -0.13305221, -0.03...","[0.010603966424999998, 0.014678564500000001, -..."
4,https://www.dice.com/jobs/detail/DevOps-Engine...,Matrix Resources,Full Time,Midtown based high tech firm has an immediate ...,Dice Id : matrixga,"Atlanta, GA",DevOps Engineer,48 minutes ago,Telecommuting not available|Travel not required,None,"configuration management, developer, linux, ma...",45efa1f6bc65acc32bbbb953a1ed13b7,"[DevOps, Project Management, Scripting, Config...","[DevOps, project management, project configura...","[Ansible, DevOps, manipulate puppets, project ...","[-0.017652616, 0.057342496, -0.13930643, -0.05...","[-0.010731407, 0.060096607, -0.1377153, -0.060...","[0.0009501145119999992, 0.033562203, -0.132539..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21995,https://www.dice.com/jobs/detail/Web-Designer-...,IAC Publishing,Full Time,Company Description We are searching for a ta...,Dice Id : 10112803,"Oakland, CA",Web Designer,3 weeks ago,Telecommuting not available|Travel not required,None,"ui/ux mobile apps, interaction design, digital...",86e27ce6b7e631e55d69d142c7d43df2,"[Python, Project Management, Data Analysis, UI

In [9]:
jobs = jobs[~jobs['avg_skill_embedding'].isna()]

In [10]:
jobs = jobs.reset_index(drop=True)

In [11]:
job_embeddings = jobs['avg_skill_embedding'].tolist()

In [12]:
resume_embeddings = resumes['avg_skill_embedding'].tolist()

In [13]:
job_embeddings = jobs['avg_skill_embedding'].tolist()

In [14]:
similarities = cosine_similarity(resume_embeddings, job_embeddings)

In [15]:
best_match_indices = similarities.argmax(axis=1)
best_match_scores = similarities.max(axis=1)

In [16]:
resumes['best_match_index'] = best_match_indices
resumes['best_match_score'] = best_match_scores
resumes['best_match_job_embedding'] = jobs.loc[best_match_indices, 'avg_skill_embedding'].values
resumes['best_match_job_skills'] = jobs.loc[best_match_indices, 'matched_skills_ordered'].values

In [17]:
resumes

,ID,Resume_str,Resume_html,Category,extracted_skills,length,matched_skills,matched_skills_ordered,skill_embeddings,skill_embeddings_ordered,avg_skill_embedding,best_match_index,best_match_score,best_match_job_embedding,best_match_job_skills
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR,"[Accounting, Ads, Advertising, Analytical skil...",5442,"[accounting, create advertisements, outdoor ad...","[accounting, company policies, conduct public ...","[-0.051119804, 0.051380936, -0.1956489, -0.000...","[-0.028823888, 0.056566164, -0.19899392, -0.00...","[0.0002473207458333337, 0.03493942622916666, -...",6143,0.928638,"[-0.011588358229999998, 0.0332843904, -0.16132...","[carry out event management, devise special pr..."
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR,"[Adobe Photoshop, ADP, Asset Management, brand...",5572,"[Adobe Photoshop, asset management, maintain c...","[Adobe Illustrator, Adobe Photoshop, asset man...","[-0.013715399, 0.01982399, -0.15967667, 0.0198...","[-0.02063493, 0.030683268, -0.16921557, 0.0126...","[0.002284706435294118, 0.01609132022352941, -0...",2571,0.945518,"[-0.015091892908333335, 0.024637934266666665, ...","[Adobe Illustrator, Adobe Photoshop, adobe cre..."
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR,"[Recruiting, FMLA/EEO/FLSA, HRIS Development, ...",7720,"[recruit members, foreign affairs policy devel...","[Microsoft Access, desktop publishing, foreign...","[-0.037161566, 0.0036097213, -0.19848743, -0.0...","[0.0076125017, 0.01415186, -0.17561796, -0.009...","[-0.0002954738888888898, 0.02081807622222222, ...",8590,0.938758,"[-0.002693496843749999, 0.03136674927875, -0.1...","[align efforts towards business development, b..."
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR,"[Type 50 wpm and 10-Key by touch, Microsoft pr...",2855,"[personnel management, customer service, resol...","[customer service, personnel management, resol...","[-0.004071229, -0.0062129777, -0.16848274, -0....","[-0.0024955925, -0.005882781, -0.17091864, -0....","[0.007837255066666667, 0.0018740463333333332, ...",17447,0.903275,"[-0.0022309643, 0.019820139285714283, -0.17098...","[communication, conflict management, customer ..."
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR,"[ADA, ADP, art, agency, benefits, Benefits Adm...",9172,"[create artwork, coaching techniques, coaching...","[Microsoft Access, Microsoft Visio, coaching t...","[0.008319367, 0.02057786, -0.17201392, 0.01393...","[4.8317004e-05, 0.005026085, -0.17149556, 0.01...","[0.00895575635652174, 0.013967274808695652, -0...",19099,0.951315,"[0.013024438820000001, 0.021138567258000004, -...","[Agile development, Python (computer programmi..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2444,99416532,RANK: SGT/E-5 NON- COMMISSIONED OFFIC...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION,"[Secret Clearance, Stock Control, Management, ...",5533,"[follow stock control instructions, operations...","[communication, follow stock control instructi...","[0.0049826736, 0.0045687603, -0.19545835, -0.0...","[0.0075103026, -0.0060989116, -0.19636096, -0....","[0.007711735449999999, 0.005379060733333334, -...",14677,0.919940,"[0.007246956881818181, 0.0223674163, -0.160678...","[analyse business requirements, communication,..."
2445,24589765,"GOVERNMENT RELATIONS, COMMUNICATIONS ...","<div class=""fontsize fontface vmargins hmargin...",AVIATION,"[arbitration, agency, budgets, Budget, continu...",7108,"[examine budgets, update budget, continuous im...","[Microsoft Visio, computer graphics, continuou...","[0.04153785, 0.038161203, -0.16648234, -0.0115...","[0.048209656, 0.025236994, -0.15883973, -0.005...","[0.013527077672727272, 0.028311003981818184, -...",16902,0.932184,"[0.01

In [18]:
resumes['true_missing'] = resumes.apply(
    lambda row: list(set(row['best_match_job_skills']) - set(row['matched_skills_ordered'])),
    axis=1
)

In [19]:
preds = resumes.apply(
    lambda row: helpers.predict_missing(row, job_embeddings, jobs, 'matched_skills', 'avg_skill_embedding'),
    axis=1
)
resumes['difference'] = preds.apply(lambda x: x["difference"])
resumes['predicted_missing'] = preds.apply(lambda x: x["skills"])
resumes['predicted_missing_id'] = preds.apply(lambda x: x["id"])
resumes['predicted_missing_embedding'] = preds.apply(lambda x: x["embeddings"])

In [20]:
scoring.found_score(resumes['true_missing'].tolist(),resumes['predicted_missing'].tolist())

np.float64(0.2506613451582933)

In [21]:
scoring.unnecessary_score(resumes['true_missing'].tolist(),resumes['predicted_missing'].tolist())

np.float64(0.26915723335690667)

In [22]:
scoring.redundant_score(resumes['predicted_missing'].tolist(), resumes['matched_skills_ordered'].tolist())

np.float64(0.04557342525901121)

In [23]:
scoring.presence_score(resumes['true_missing'].tolist(), resumes['predicted_missing'].tolist())

0.8775010208248265

In [24]:
resumes['best_match_score'].mean()

np.float64(0.917210003230938)

In [ ]:
index = 15
print("Job Description")
print(resumes['best_match_job_skills'].iloc[index])
print()
print("Sample Resume")
print(resumes['matched_skills_ordered'].iloc[index])

In [ ]:
true_missing = set(resumes['best_match_job_skills'].iloc[index])-set(resumes['matched_skills_ordered'].iloc[index])

In [ ]:
set(resumes['true_missing'].iloc[index])

In [ ]:
len(true_missing)

In [ ]:
diff = resumes['best_match_job_embedding'].iloc[index] - resumes['avg_skill_embedding'].iloc[index]

In [ ]:
diffs = cosine_similarity([diff], job_embeddings)

In [ ]:
diffs.argmax()

In [ ]:
predicted_missing = set(jobs['matched_skills_ordered'].iloc[diffs.argmax()])

In [ ]:
predicted_missing

In [ ]:
true_missing.intersection(predicted_missing)

In [ ]:
predicted_missing-true_missing

In [ ]:
print(len(true_missing.intersection(predicted_missing)))
print(len(true_missing-predicted_missing))

In [ ]:
true_missing.symmetric_difference(predicted_missing)

In [ ]:
resumes['true_missing'].tolist()